In [1]:
import subprocess
import os
import csv
import pandas as pd
from bayes_opt import BayesianOptimization
from pathlib import Path
import numpy as np
from simulation_functions import call_vot_simulation, intialise_java, call_stt_simulation,stats_test, call_stt_unified_simulation, STT_INIT_POINTS, STT_N_ITER, VOT_INIT_POINTS, VOT_N_ITER
from scipy import stats

VOT config

In [2]:
#VOT config
defaultAlpha_bounds  = (-5,5)
default_beta_bounds = (-1, -0.001)
vot_pbounds = {'alphaWalk' :(0,0) ,
           'alphaBike' : defaultAlpha_bounds,
           'alphaCarDriver' : (-10,10),
           'alphaCarPassenger' : defaultAlpha_bounds,
           'alphaBus' : defaultAlpha_bounds,
           'alphaTrain' : defaultAlpha_bounds,
           'betaChangesTransport': default_beta_bounds,
           'betaCostLow':(-1.0,-0.2) ,
           'betaCostMed': (-0.2,-0.2),
           'betaCostHigh': (-0.2,-0.01),
           'weightWalk': (1,3),
           'weightWait': (1,3),
           'weightFeeder': (1,3), 
           'weightVotCosts' : (1,1),
           'weightTangibleCosts' : (1,1)
           } 

STT config

In [3]:
stt_pbounds = {
    'alphaWalk' :(0,0) ,
    'alphaBike' : defaultAlpha_bounds,
    'alphaCarDriver' : defaultAlpha_bounds,
    'alphaCarPassenger' : defaultAlpha_bounds,
    'alphaBus' : defaultAlpha_bounds,
    'alphaTrain' : defaultAlpha_bounds,
    'betaTimeWalk' : (-0.04,-0.04),
    'betaTimeBike' : (-0.03,-0.03),
    'betaTimeCarDriver': (-0.02,-0.02),
    'betaTimeCarPassenger' : (-0.02,-0.02),
    'betaTimeBus' : (-0.02,-0.02),
    'betaTimeTrain' : (-0.02,-0.02),
    'betaCostCarDriver': (-0.15,-0.15),
    'betaCostCarPassenger' : (-0.15,-0.15),
    'betaCostBus': (-0.1,-0.1),
    'betaCostTrain': (-0.1,-0.1),
    'betaTimeWalkTransport': (-0.03,-0.03),
    'betaChangesTransport': (-0.3,-0.3)}

STT Unified Config

In [4]:
#This is litterally the same, just different coefficients. 
stt_pbounds_default_coefficients = {
    'alphaWalk' : (0,0),
    'alphaBike' :defaultAlpha_bounds,
    'alphaCarDriver': defaultAlpha_bounds,
    'alphaCarPassenger':defaultAlpha_bounds,
    'alphaBus':defaultAlpha_bounds,
    'alphaTrain' :defaultAlpha_bounds,
    'betaChangesTransport': (-0.03,-0.03),
    'betaTimeWalkTransport' :  (-0.3,-0.3),
    'betaCostLow' :(-0.2,-0.2),
    'betaCostMed' : (-0.2,-0.2),
    'betaCostHigh': (-0.2,-0.2),
    'votCommuteWalk': (-0.04,-0.04),
    'votCommuteBike': (-0.03,-0.03),
    'votCommuteCar': (-0.02,-0.02),
    'votCommutePassenger': (-0.02,-0.02),
    'votCommuteBus': (-0.02,-0.02),
    'votCommuteTrain':(-0.02,-0.02),
    'votOtherWalk': (-0.04,-0.04),
    'votOtherBike':(-0.03,-0.03),
    'votOtherCar': (-0.02,-0.02),
    'votOtherPassenger':(-0.02,-0.02),
    'votOtherBus':(-0.02,-0.02),
    'votOtherTrain':(-0.02,-0.02)}

stt_pbounds_vot_coefficients = {
    'alphaWalk' : (0,0),
    'alphaBike' :defaultAlpha_bounds,
    'alphaCarDriver': defaultAlpha_bounds,
    'alphaCarPassenger':defaultAlpha_bounds,
    'alphaBus':defaultAlpha_bounds,
    'alphaTrain' :defaultAlpha_bounds,
    'betaChangesTransport': (-0.03,-0.03),
    'betaTimeWalkTransport' :  (-0.3,-0.3),
    'betaCostLow' :(-0.2,-0.2),
    'betaCostMed' : (-0.2,-0.2),
    'betaCostHigh': (-0.2,-0.2),
    'votCommuteWalk': (-0.0529,-0.0529),
    'votCommuteBike': (-0.0339,-0.0339),
    'votCommuteCar': (-0.359,-0.359),
    'votCommutePassenger': (-0.359,-0.359),
    'votCommuteBus': (-0.0254,-0.0254),
    'votCommuteTrain':(-0.0402,-0.0402),
    'votOtherWalk': (-0.0392,-0.0392),
    'votOtherBike':(-0.0348,-0.0348),
    'votOtherCar': (-0.032,-0.032),
    'votOtherPassenger':(-0.032,-0.032),
    'votOtherBus':(-0.0222,-0.0222),
    'votOtherTrain':(-0.0288,-0.0288)}

In [5]:
stt_gen_synth_pop_baseline = [1.538359177843295,
 1.5295917791690679,
 1.5348367663998912,
 1.5345692372273643,
 1.5331592512372867,
 1.5217034952544148,
 1.5266326185511052,
 1.5334169124971047,
 1.5360137686306476,
 1.5339017221565026,
 1.5362097781749502,
 1.5268521109958817,
 1.5339477732745261,
 1.5272698871615034,
 1.5366142065675472,
 1.5363138577063769,
 1.5296998668830282,
 1.5197286106223606,
 1.5389714190118928,
 1.5313439434581202,
 1.537382448574118,
 1.5343533532165867,
 1.533794502058735,
 1.5379044402275153,
 1.540476837032657,
 1.5442550528088403,
 1.5314370912595778,
 1.5282256003186678,
 1.529788503142131,
 1.5278669293628444]

stt_base_pop_baseline = [8.078856646164647,
 8.069551887294185,
 8.075341126236319,
 8.077668179308883,
 8.072089261165532,
 8.08434725328873,
 8.070135626168561,
 8.07003999884651,
 8.074229639153051,
 8.073871153762488,
 8.076644709455191,
 8.067903638080667,
 8.074645258436167,
 8.07695697572408,
 8.069666279765226,
 8.073238056549226,
 8.075901298340282,
 8.074258667977556,
 8.066514722376704,
 8.072437841187186,
 8.07148870370448,
 8.069260779586159,
 8.067953580518305,
 8.074462078074248,
 8.072890298665042,
 8.076124680057342,
 8.07104170586535,
 8.066528599568002,
 8.070791356185456,
 8.081042173765757]

rq_3_4_results = [1.3753081319569687,
 1.3742488303399583,
 1.368884010723444,
 1.3901922800580793,
 1.361084756061178,
 1.3775440627186848,
 1.3790867291461066,
 1.3727054733959452,
 1.365534949443592,
 1.3863956306982672,
 1.369942893561297,
 1.3627206632717621,
 1.3818975650700178,
 1.3661656227329662,
 1.3582229687456173,
 1.3829006610569607,
 1.371984528786159,
 1.379002606109665,
 1.3698603823393452,
 1.3873749242135547,
 1.376056809933175,
 1.388874581056718,
 1.3668109468143579,
 1.3580711757372352,
 1.3754509132818438,
 1.3793868968859846,
 1.3736371235050056,
 1.3867188912473,
 1.3659524784652293,
 1.3679807519092755]

rq_4_1_results = [4.107192103461534,
 4.089931333707253,
 4.104172307549402,
 4.099819286067584,
 4.092147380546496,
 4.113514427796199,
 4.089557072905693,
 4.104245321688032,
 4.0993655240625015,
 4.103798878625523,
 4.101218299896193,
 4.07609955762409,
 4.112416916315092,
 4.105474449116669,
 4.087083678727698,
 4.100771285685858,
 4.09217321538128,
 4.092633707903322,
 4.095278441379898,
 4.106089861485245,
 4.101872030124147,
 4.1000343691216345,
 4.082920879017399,
 4.079481500744014,
 4.098486818062991,
 4.092394595584255,
 4.087475965220166,
 4.086071301758151,
 4.0968919765924285,
 4.078628686540781]

In [6]:
intialise_java()

Java Version Output:
 openjdk version "11.0.20.1" 2023-08-24 LTS
OpenJDK Runtime Environment Microsoft-8297088 (build 11.0.20.1+1-LTS)
OpenJDK 64-Bit Server VM Microsoft-8297088 (build 11.0.20.1+1-LTS, mixed mode)



In [ ]:

# rq_4_1_config = {
#     "config_file": "src/main/resources/config_DHZW_rq4.toml",
#     "parameterset": "src/main/resources/baseline_parameterset/parameterset_rq4.csv",
#     "parameterset_write_folder" : "../7-simulation-Sim-2APL/",
#     "output_path": 'src/main/resources/baseline_parameterset/output-rq4.csv',
#     "other_args" : "--use_random_seed true"
# }
# rq4_1_optimiser = BayesianOptimization(
#     f=lambda **params: call_vot_simulation(config=rq_4_1_config, **params),
#     pbounds=vot_pbounds,
#     random_state=1,
# )

# rq4_1_optimiser.maximize(
#     init_points=VOT_INIT_POINTS,
#     n_iter=VOT_N_ITER
# )

|   iter    |  target   | alphaWalk | alphaBike | alphaC... | alphaC... | alphaBus  | alphaT... | betaCh... | betaCo... | betaCo... | betaCo... | weight... | weight... | weight... | weight... | weight... |
-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
| 1         | -33.35300 | 0.0       | 2.2032449 | -9.997712 | -1.976674 | -3.532441 | -4.076614 | -0.813926 | -0.723551 | -0.2      | -0.097624 | 1.8383890 | 2.3704390 | 1.4089044 | 1.0       | 1.0       |
| 2         | -12.63315 | 0.0       | -0.826951 | 1.1737965 | -3.596130 | -3.018985 | 3.0074456 | -0.032706 | -0.749260 | -0.2      | -0.033486 | 2.7892133 | 1.1700884 | 1.0781095 | 1.0       | 1.0       |
| 3         | -8.335236 | 0.0       | -0.788923 | 9.1577906 | 0.3316528 | 1.9187711 | -1.844843 | -0.314185 | -0.332299 | -0.2      | -0.057472 | 2.9777221 | 2.4963313 | 1.5608

In [ ]:
# rq_1_config_no_seed = rq_4_1_config
# rq_1_config_no_seed["other_args"] = "--use_random_seed false"
# rq_4_1_results = []
# for i in range(0,30):
#     rq_4_1_results.append(- call_vot_simulation(config=rq_1_config_no_seed,**rq4_1_optimiser.max['params']))


In [ ]:
# rq_4_1_results

[4.107192103461534,
 4.089931333707253,
 4.104172307549402,
 4.099819286067584,
 4.092147380546496,
 4.113514427796199,
 4.089557072905693,
 4.104245321688032,
 4.0993655240625015,
 4.103798878625523,
 4.101218299896193,
 4.07609955762409,
 4.112416916315092,
 4.105474449116669,
 4.087083678727698,
 4.100771285685858,
 4.09217321538128,
 4.092633707903322,
 4.095278441379898,
 4.106089861485245,
 4.101872030124147,
 4.1000343691216345,
 4.082920879017399,
 4.079481500744014,
 4.098486818062991,
 4.092394595584255,
 4.087475965220166,
 4.086071301758151,
 4.0968919765924285,
 4.078628686540781]

Sanity Check

In [21]:
# Test similarity, should have RMSE of about 1.48 
results = {'alphaWalk': np.float64(0.0),
 'alphaBike': np.float64(-0.707348373421328),
 'alphaCarDriver': np.float64(4.086472528950937),
 'alphaCarPassenger': np.float64(2.874633051402859),
 'alphaBus': np.float64(0.33044361294562175),
 'alphaTrain': np.float64(1.1637548184898898)}




sanity_check =  stt_pbounds_default_coefficients
sanity_check['alphaWalk'] = (0,0)
sanity_check['alphaBike'] =  (-0.707348373421328,-0.707348373421328)
sanity_check['alphaCarDriver'] =  (4.086472528950937,4.086472528950937)
sanity_check['alphaCarPassenger'] = (2.874633051402859,2.874633051402859)
sanity_check['alphaBus'] = (0.33044361294562175,0.33044361294562175)
sanity_check['alphaTrain'] = (1.1637548184898898,1.1637548184898898)

#need to run sim with this...
sanity_check_config = {
    "config_file": "src/main/resources/config_DHZW_rq4-1-sanity-check.toml",
    "parameterset": "src/main/resources/baseline_parameterset/parameterset_rq4-1-sanity-check.csv",
    "parameterset_write_folder" : "../7-simulation-Sim-2APL/",
    "output_path": 'src/main/resources/baseline_parameterset/parameterset_rq4-1-sanity-check.csv',
    "other_args" : "--use_random_seed true"
}

rq1_2_optimiser = BayesianOptimization(
    f=lambda **params: call_stt_unified_simulation(config=sanity_check_config, **params),
    pbounds=sanity_check,
    random_state=1,
)

rq1_2_optimiser.maximize(
    init_points=1,
    n_iter=1

)

|   iter    |  target   | alphaWalk | alphaBike | alphaC... | alphaC... | alphaBus  | alphaT... | betaCh... | betaTi... | betaCo... | betaCo... | betaCo... | votCom... | votCom... | votCom... | votCom... | votCom... | votCom... | votOth... | votOth... | votOth... | votOth... | votOth... | votOth... |
-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
| 1         | -8.902034 | 0.0       | -0.707348 | 4.0864725 | 2.8746330 | 0.3304436 | 1.1637548 | -0.03     | -0.3      | -0.2      | -0.2      | -0.2      | -0.04     | -0.03     | -0.02     | -0.02     | -0.02     | -0.02     | -0.04     | -0.03     | -0.02     | -0.02     | -0.02     | -0.02     |
| 2         | -8.902034 | 0.0       | -0.707348 | 4.0864725 | 2.8746330 | 0.3304436 | 1.163754

Of course, not the same as the cost coefficients aren't uniform......

E4.2

In [7]:
e_4_2_config = {
    "config_file": "src/main/resources/config_DHZW_rq4-2.toml",
    "parameterset": "src/main/resources/baseline_parameterset/parameterset_rq4-2.csv",
    "parameterset_write_folder" : "../7-simulation-Sim-2APL/",
    "output_path": 'src/main/resources/baseline_parameterset/parameterset_rq4-2.csv',
    "other_args" : "--use_random_seed true"
}

e_4_2_optimiser = BayesianOptimization(
    f=lambda **params: call_stt_unified_simulation(config=e_4_2_config, **params),
    pbounds=stt_pbounds_default_coefficients,
    random_state=1,
)

e_4_2_optimiser.maximize(
    init_points=STT_INIT_POINTS,
    n_iter=STT_N_ITER

)

|   iter    |  target   | alphaWalk | alphaBike | alphaC... | alphaC... | alphaBus  | alphaT... | betaCh... | betaTi... | betaCo... | betaCo... | betaCo... | votCom... | votCom... | votCom... | votCom... | votCom... | votCom... | votOth... | votOth... | votOth... | votOth... | votOth... | votOth... |
-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
| 1         | -33.55815 | 0.0       | 2.2032449 | -4.998856 | -1.976674 | -3.532441 | -4.076614 | -0.03     | -0.3      | -0.2      | -0.2      | -0.2      | -0.04     | -0.03     | -0.02     | -0.02     | -0.02     | -0.02     | -0.04     | -0.03     | -0.02     | -0.02     | -0.02     | -0.02     |
| 2         | -26.72601 | 0.0       | 3.7638915 | 3.9460666 | -4.149557 | -4.609452 | -3.30169

In [8]:
e_4_2_config_no_seed = e_4_2_config
e_4_2_config_no_seed["other_args"] = "--use_random_seed false"
e_4_2_results = []
for i in range(0,30):
    e_4_2_results.append(- call_stt_unified_simulation(config=e_4_2_config_no_seed,**e_4_2_optimiser.max['params']))

In [9]:
e_4_2_results

[4.246199271010122,
 4.2597496931167305,
 4.252352226515116,
 4.245098091001778,
 4.239709475044977,
 4.257492287706117,
 4.266831408898299,
 4.256758458140026,
 4.2430470226189305,
 4.2603814590627,
 4.263919093672596,
 4.259879582740013,
 4.245113146535995,
 4.244550688283676,
 4.218479807723878,
 4.257842037829524,
 4.271262509672729,
 4.270920009734737,
 4.229471685878263,
 4.256782996518147,
 4.257530128705013,
 4.251328252328521,
 4.2547073115722664,
 4.264381656533188,
 4.2598724144333735,
 4.268623533675737,
 4.257014044785765,
 4.246766804199332,
 4.25440461901751,
 4.252799502471978]

E4.3

In [10]:
STT_INC = STT_INIT_POINTS+20
STT_ITER_INC = STT_N_ITER + 60

In [11]:
e_4_3_config = {
    "config_file": "src/main/resources/config_DHZW_rq4-3.toml",
    "parameterset": "src/main/resources/baseline_parameterset/parameterset_rq4-3.csv",
    "parameterset_write_folder" : "../7-simulation-Sim-2APL/",
    "output_path": 'src/main/resources/baseline_parameterset/parameterset_rq4-3.csv',
    "other_args" : "--use_random_seed true"
}

e_4_3_pbounds = stt_pbounds_default_coefficients 

e_4_3_pbounds['betaCostLow'] = (-1.0,-0.2)
e_4_3_pbounds['betaCostHigh'] = (-0.2,-0.01)
e_4_3_optimiser = BayesianOptimization(
    f=lambda **params: call_stt_unified_simulation(config=e_4_3_config, **params),
    pbounds=e_4_3_pbounds,
    random_state=1,
)

e_4_3_optimiser.maximize(
    init_points=STT_INC,
    n_iter=STT_ITER_INC

)

|   iter    |  target   | alphaWalk | alphaBike | alphaC... | alphaC... | alphaBus  | alphaT... | betaCh... | betaTi... | betaCo... | betaCo... | betaCo... | votCom... | votCom... | votCom... | votCom... | votCom... | votCom... | votOth... | votOth... | votOth... | votOth... | votOth... | votOth... |
-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
| 1         | -33.56766 | 0.0       | 2.2032449 | -4.998856 | -1.976674 | -3.532441 | -4.076614 | -0.03     | -0.3      | -0.682586 | -0.2      | -0.120353 | -0.04     | -0.03     | -0.02     | -0.02     | -0.02     | -0.02     | -0.04     | -0.03     | -0.02     | -0.02     | -0.02     | -0.02     |
| 2         | -27.02434 | 0.0       | 3.7638915 | 3.9460666 | -4.149557 | -4.609452 | -3.30169

In [12]:
e_4_3_config_no_seed = e_4_3_config
e_4_3_config_no_seed["other_args"] = "--use_random_seed false"
e_4_3_results = []
for i in range(0,30):
    e_4_3_results.append(- call_stt_unified_simulation(config=e_4_3_config_no_seed,**e_4_3_optimiser.max['params']))

In [13]:
e_4_3_results

[4.22065794949561,
 4.219743652238627,
 4.195033964850989,
 4.210123790423449,
 4.203744033197774,
 4.2234309606881295,
 4.2255301298006644,
 4.204327395418513,
 4.205845183076908,
 4.205900047806511,
 4.240529736275672,
 4.196735556728009,
 4.212907440664713,
 4.213987362593462,
 4.19452706109437,
 4.206650858170659,
 4.2261596250064075,
 4.211983203158505,
 4.180619769446487,
 4.195807296858411,
 4.196803958106656,
 4.194138127079904,
 4.211085083631379,
 4.1818716550194805,
 4.197383048380087,
 4.208600883789695,
 4.214256529739772,
 4.218403529902263,
 4.224971554091382,
 4.216784759783043]

E4.4

In [14]:
e_4_4_config = {
    "config_file": "src/main/resources/config_DHZW_rq4-4.toml",
    "parameterset": "src/main/resources/baseline_parameterset/parameterset_rq4-4.csv",
    "parameterset_write_folder" : "../7-simulation-Sim-2APL/",
    "output_path": 'src/main/resources/baseline_parameterset/parameterset_rq4-4.csv',
    "other_args" : "--use_random_seed true"
}
e_4_4_optimiser = BayesianOptimization(
    f=lambda **params: call_stt_unified_simulation(config=e_4_4_config, **params),
    pbounds=stt_pbounds_vot_coefficients,
    random_state=1,
)

e_4_4_optimiser.maximize(
    init_points=STT_INIT_POINTS,
    n_iter=STT_N_ITER

)

|   iter    |  target   | alphaWalk | alphaBike | alphaC... | alphaC... | alphaBus  | alphaT... | betaCh... | betaTi... | betaCo... | betaCo... | betaCo... | votCom... | votCom... | votCom... | votCom... | votCom... | votCom... | votOth... | votOth... | votOth... | votOth... | votOth... | votOth... |
-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
| 1         | -33.57192 | 0.0       | 2.2032449 | -4.998856 | -1.976674 | -3.532441 | -4.076614 | -0.03     | -0.3      | -0.2      | -0.2      | -0.2      | -0.0529   | -0.0339   | -0.359    | -0.359    | -0.0254   | -0.0402   | -0.0392   | -0.0348   | -0.032    | -0.032    | -0.0222   | -0.0288   |
| 2         | -29.01405 | 0.0       | 3.7638915 | 3.9460666 | -4.149557 | -4.609452 | -3.30169

In [15]:
e_4_4_config_no_seed = e_4_4_config
e_4_4_config_no_seed["other_args"] = "--use_random_seed false"
e_4_4_results = []
for i in range(0,30):
    e_4_4_results.append(- call_stt_unified_simulation(config=e_4_4_config_no_seed,**e_4_4_optimiser.max['params']))

In [16]:
e_4_4_results

[4.40307520368265,
 4.431346513692629,
 4.402647432229831,
 4.413569612958918,
 4.403900941974497,
 4.407449412718828,
 4.411399716371597,
 4.413187562656271,
 4.403385370699314,
 4.410968368713014,
 4.409601216647042,
 4.40721243732848,
 4.404721153614437,
 4.415089915602482,
 4.408262294905312,
 4.4322123876525765,
 4.410843718143001,
 4.408164114306393,
 4.40958292418684,
 4.433190978233629,
 4.410426460696309,
 4.418753140801254,
 4.406858104603789,
 4.404416905564994,
 4.404959720449975,
 4.422955448492967,
 4.418787216716721,
 4.399046018092787,
 4.431328425005369,
 4.404162272893112]

E4.5

In [17]:
e_4_5_config = {
    "config_file": "src/main/resources/config_DHZW_rq4-5.toml",
    "parameterset": "src/main/resources/baseline_parameterset/parameterset_rq4-5.csv",
    "parameterset_write_folder" : "../7-simulation-Sim-2APL/",
    "output_path": 'src/main/resources/baseline_parameterset/parameterset_rq4-5.csv',
    "other_args" : "--use_random_seed true"
}

#Use vot, but allow modifying the cost coefficients
e_4_5_pbounds = stt_pbounds_vot_coefficients 

e_4_5_pbounds['betaCostLow'] = (-1.0,-0.2)
e_4_5_pbounds['betaCostHigh'] = (-0.2,-0.01)
e_4_5_optimiser = BayesianOptimization(
    f=lambda **params: call_stt_unified_simulation(config=e_4_5_config, **params),
    pbounds=e_4_5_pbounds,
    random_state=1,
)

e_4_5_optimiser.maximize(
    init_points=STT_INC,
    n_iter=STT_ITER_INC

)

|   iter    |  target   | alphaWalk | alphaBike | alphaC... | alphaC... | alphaBus  | alphaT... | betaCh... | betaTi... | betaCo... | betaCo... | betaCo... | votCom... | votCom... | votCom... | votCom... | votCom... | votCom... | votOth... | votOth... | votOth... | votOth... | votOth... | votOth... |
-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
| 1         | -33.58312 | 0.0       | 2.2032449 | -4.998856 | -1.976674 | -3.532441 | -4.076614 | -0.03     | -0.3      | -0.682586 | -0.2      | -0.120353 | -0.0529   | -0.0339   | -0.359    | -0.359    | -0.0254   | -0.0402   | -0.0392   | -0.0348   | -0.032    | -0.032    | -0.0222   | -0.0288   |
| 2         | -29.19321 | 0.0       | 3.7638915 | 3.9460666 | -4.149557 | -4.609452 | -3.30169

In [18]:
e_4_5_config_no_seed = e_4_5_config
e_4_5_config_no_seed["other_args"] = "--use_random_seed false"
e_4_5_results = []
for i in range(0,30):
    e_4_5_results.append(- call_stt_unified_simulation(config=e_4_5_config_no_seed,**e_4_5_optimiser.max['params']))

In [19]:
e_4_5_results

[4.408635883930507,
 4.410279839282796,
 4.413114604473231,
 4.411629305482592,
 4.416092093183844,
 4.39682402160628,
 4.4080797961887175,
 4.408924657874151,
 4.429139460553077,
 4.408365391557447,
 4.420629324318037,
 4.414464784538914,
 4.383010186385405,
 4.427775037486159,
 4.401491287424597,
 4.419851892095862,
 4.396528527742302,
 4.410484937799134,
 4.4234943772154764,
 4.400448661635156,
 4.412093051882678,
 4.430497578206732,
 4.415113949784371,
 4.383011140376671,
 4.407845152801464,
 4.409206283075956,
 4.411260924279036,
 4.414408788172662,
 4.428763538736332,
 4.41028473599324]

E5.6